In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.metrics import mean_squared_error
import imageio_ffmpeg

# -------------------------------------------------------------
# Step 0 
# -------------------------------------------------------------
plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

# -------------------------------------------------------------
# Step 1 (Train: 70%, Val: 15%, Test: 15%)
# -------------------------------------------------------------
np.random.seed(42)

N = 300
x_all = np.linspace(-3.0, 3.0, N)
y_clean = np.sin(3.5 * x_all) + 0.3 * np.cos(7.0 * x_all) + 0.5 * x_all**2 - 0.1 * x_all**3
y_all = y_clean + np.random.normal(0, 0.30, size=N)


middle_indices = np.arange(1, N - 1)
np.random.shuffle(middle_indices)

n_train_mid = int(0.70 * N) - 2
n_val = int(0.15 * N)

train_idx = np.sort(np.concatenate(([0, N - 1], middle_indices[:n_train_mid])))
val_idx = middle_indices[n_train_mid : n_train_mid + n_val]
test_idx = middle_indices[n_train_mid + n_val :]

x_train, y_train = x_all[train_idx], y_all[train_idx]
x_val, y_val = x_all[val_idx], y_all[val_idx]
x_test, y_test = x_all[test_idx], y_all[test_idx]

x_dense = np.linspace(-3.0, 3.0, 400)
y_clean_dense = np.sin(3.5 * x_dense) + 0.3 * np.cos(7.0 * x_dense) + 0.5 * x_dense**2 - 0.1 * x_dense**3

# -------------------------------------------------------------
# Step 2
# -------------------------------------------------------------
def tricube_kernel(distances, max_dist):
    
    u = np.clip(distances / max_dist, 0, 1)
    return (1 - u**3)**3

def loess_single_point(x_eval, x_tr, y_tr, span=0.3, degree=1):
    
    distances = np.abs(x_tr - x_eval)
    k = max(int(np.ceil(span * len(x_tr))), 2)
    max_dist = np.partition(distances, k - 1)[k - 1]
    
    weights = tricube_kernel(distances, max_dist)
    
    # WLS 
    if degree == 1:
        X = np.column_stack([np.ones_like(x_tr), x_tr - x_eval])
    else:
        X = np.column_stack([np.ones_like(x_tr), x_tr - x_eval, (x_tr - x_eval)**2])
        
    W = np.diag(weights)
    
    try:
        # Weighted Least Squares
        beta = np.linalg.lstsq(X.T @ W @ X, X.T @ W @ y_tr, rcond=None)[0]
        y_pred = beta[0]
    except np.linalg.LinAlgError:
        y_pred = np.mean(y_tr)
        beta = [y_pred, 0]
        
    return y_pred, weights, beta, max_dist

def predict_loess_full(x_eval_arr, x_tr, y_tr, span=0.3, degree=1):
    
    preds = [loess_single_point(x, x_tr, y_tr, span, degree)[0] for x in x_eval_arr]
    return np.array(preds)

# -------------------------------------------------------------
# Step 3
# -------------------------------------------------------------
spans_to_test = [0.08, 0.15, 0.25, 0.40, 0.60]
val_mses = []
for s in spans_to_test:
    val_preds = predict_loess_full(x_val, x_train, y_train, span=s)
    val_mses.append(mean_squared_error(y_val, val_preds))

best_span = spans_to_test[np.argmin(val_mses)]

# -------------------------------------------------------------
# Step 4
# -------------------------------------------------------------


frames_sliding = 60
total_frames = 10 + 10 + frames_sliding + 20 + 20

fig, ax = plt.subplots(figsize=(11, 6.5), dpi=100)

def animate(frame_idx):
    ax.clear()


    ax.scatter(x_train, y_train, color='blue', alpha=0.35, s=25, label='Train Data (70%)', zorder=3)
    ax.scatter(x_val, y_val, color='red', alpha=0.55, s=35, marker='o', label='Val Data (15%)', zorder=4)
    ax.scatter(x_test, y_test, color='gray', alpha=0.55, s=35, marker='s', label='Test Data (15%)', zorder=4)
    ax.plot(x_dense, y_clean_dense, 'k--', alpha=0.25, linewidth=1.5, label='True Function')

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    if frame_idx < 10:
        p = np.poly1d(np.polyfit(x_train, y_train, 1))
        ax.plot(x_dense, p(x_dense), color='red', linewidth=3.5, label='Global Linear Fit', zorder=6)
        ax.set_title("Stage 1: Global Linear Regression (High Bias)", fontsize=13, fontweight='bold', color='darkred')
        
        val_mse = mean_squared_error(y_val, p(x_val))
        info_str = f"Stage: Global Linear Fit\nVal MSE: {val_mse:.4f}\nStatus: Severe Underfitting"
        box_bg, edge_c = '#f8d7da', 'darkred'

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    elif 10 <= frame_idx < 20:
        x_eval = 0.0
        y_p, weights, beta, r_val = loess_single_point(x_eval, x_train, y_train, span=0.25)
        
        ax.axvline(x=x_eval, color='darkorange', linestyle='--', linewidth=2, label='Evaluation Point ($x_0$)', zorder=5)
        ax.axvspan(x_eval - r_val, x_eval + r_val, color='orange', alpha=0.15, label='Local Bandwidth Window')
        
        scatter = ax.scatter(x_train, y_train, c=weights, cmap='YlOrRd', s=20 + weights*100, zorder=6)
        
        ax.set_title("Stage 2: Local Weighted Regression (LOESS Window Concept)", fontsize=13, fontweight='bold', color='darkorange')
        info_str = (
            f"Concept: Local Weighting around $x_0 = {x_eval:.1f}$\n"
            f"Kernel: Tricube $W(x) = (1 - |u|^3)^3$\n"
            f"Status: Fitting WLS in Local Neighborhood"
        )
        box_bg, edge_c = '#fff3cd', 'darkorange'

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    elif 20 <= frame_idx < (20 + frames_sliding):
        slide_step = frame_idx - 20
        eval_indices = np.linspace(0, len(x_dense) - 1, frames_sliding, dtype=int)
        curr_idx = eval_indices[slide_step]
        x_eval = x_dense[curr_idx]
        

        y_p, weights, beta, r_val = loess_single_point(x_eval, x_train, y_train, span=0.25)
        

        x_history = x_dense[:curr_idx+1]
        y_history = predict_loess_full(x_history, x_train, y_train, span=0.25)
        ax.plot(x_history, y_history, color='darkgreen', linewidth=3.5, label='Constructed LOESS Curve', zorder=7)
        

        ax.axvline(x=x_eval, color='darkorange', linestyle='--', linewidth=1.8, zorder=5)
        ax.axvspan(x_eval - r_val, x_eval + r_val, color='orange', alpha=0.15)
        
        x_loc = np.linspace(max(-3, x_eval - r_val), min(3, x_eval + r_val), 30)
        y_loc = beta[0] + beta[1] * (x_loc - x_eval)
        ax.plot(x_loc, y_loc, color='red', linewidth=3.0, zorder=8, label='Local Linear Fit')
        
        # هایلایت نقاط با سایز متناسب با وزن
        ax.scatter(x_train, y_train, c=weights, cmap='YlOrRd', s=15 + weights*80, zorder=6)
        
        ax.set_title(f"Stage 3: Sliding Window Fitting | Point $x_0 = {x_eval:.2f}$", fontsize=13, fontweight='bold', color='darkorange')
        info_str = (
            f"Active Evaluation Point $x_0$: {x_eval:.2f}\n"
            f"Local Estimate $Y$: {y_p:.3f}\n"
            f"Status: Sliding & Fitting Local Weighted Least Squares"
        )
        box_bg, edge_c = '#fff3cd', 'darkorange'

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    elif (20 + frames_sliding) <= frame_idx < (20 + frames_sliding + 20):
        span_idx = (frame_idx - (20 + frames_sliding)) // 4
        span_idx = min(span_idx, len(spans_to_test) - 1)
        curr_span = spans_to_test[span_idx]
        
        y_dense_span = predict_loess_full(x_dense, x_train, y_train, span=curr_span)
        val_mse_curr = mean_squared_error(y_val, predict_loess_full(x_val, x_train, y_train, span=curr_span))
        
        ax.plot(x_dense, y_dense_span, color='purple', linewidth=3.5, label=f'LOESS (Span = {curr_span:.2f})', zorder=7)
        ax.set_title(f"Stage 4: Hyperparameter Tuning | Bandwidth Span = {curr_span:.2f}", fontsize=13, fontweight='bold', color='purple')
        
        status_txt = "Overfitting (Too Wiggly)" if curr_span < 0.15 else ("Underfitting (Too Smooth)" if curr_span > 0.4 else "Good Balance")
        info_str = (
            f"Span (Neighborhood Ratio): {curr_span:.2f}\n"
            f"Validation MSE: {val_mse_curr:.4f}\n"
            f"Status: {status_txt}"
        )
        box_bg, edge_c = '#e2d9f3', 'purple'

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    else:
        y_dense_best = predict_loess_full(x_dense, x_train, y_train, span=best_span)
        tr_mse = mean_squared_error(y_train, predict_loess_full(x_train, x_train, y_train, span=best_span))
        v_mse = mean_squared_error(y_val, predict_loess_full(x_val, x_train, y_train, span=best_span))
        ts_mse = mean_squared_error(y_test, predict_loess_full(x_test, x_train, y_train, span=best_span))
        
        ax.plot(x_dense, y_dense_best, color='darkgreen', linewidth=4.0, label=f'Best LOESS Model (Span={best_span:.2f})', zorder=8)
        ax.set_title("★ FINAL MODEL ★ Optimal Local Regression (LOESS/LOWESS)", fontsize=12, fontweight='bold', color='darkgreen')
        
        info_str = (
            f"FINAL LOESS MODEL SELECTION (Span = {best_span:.2f})\n"
            f"Train MSE: {tr_mse:.4f} | Val MSE: {v_mse:.4f}\n"
            f"--> TEST MSE: {ts_mse:.4f} <--\n"
            f"Status: Best Non-Parametric Generalization!"
        )
        box_bg, edge_c = '#d1e7dd', 'darkgreen'

    ax.set_ylim(-3.5, 6.0)
    ax.set_xlabel('X', fontsize=11)
    ax.set_ylabel('Y', fontsize=11)
    ax.legend(loc='upper right', fontsize=8.5, framealpha=0.95, edgecolor='gray')
    ax.grid(True, linestyle='--', alpha=0.4)

    ax.text(0.48, 0.04, info_str, transform=ax.transAxes, fontsize=10.0, fontweight='bold',
            verticalalignment='bottom', bbox=dict(boxstyle='round,pad=0.5', facecolor=box_bg, alpha=0.95, edgecolor=edge_c, linewidth=2.0))

# -------------------------------------------------------------
# -------------------------------------------------------------
ani = animation.FuncAnimation(fig, animate, frames=total_frames, interval=100, repeat=True)
writer = animation.FFMpegWriter(fps=10, bitrate=2500)
output_filename = 'loess_local_regression.mp4'

ani.save(output_filename, writer=writer)
plt.close()

print(f"ویدیو با موفقیت ساخته شد: '{output_filename}'")

ویدیو با موفقیت ساخته شد: 'loess_local_regression.mp4'


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.metrics import mean_squared_error

# -------------------------------------------------------------
# -------------------------------------------------------------
np.random.seed(101) 

N = 320
x_all = np.linspace(-4.0, 4.0, N)


y_clean = 2.0 * np.sin(2.2 * x_all) * np.exp(-0.15 * x_all**2) + 0.4 * np.cos(5.5 * x_all) + 0.3 * x_all
y_all = y_clean + np.random.normal(0, 0.28, size=N)


middle_indices = np.arange(1, N - 1)
np.random.shuffle(middle_indices)

n_train_mid = int(0.70 * N) - 2
n_val = int(0.15 * N)

train_idx = np.sort(np.concatenate(([0, N - 1], middle_indices[:n_train_mid])))
val_idx = middle_indices[n_train_mid : n_train_mid + n_val]
test_idx = middle_indices[n_train_mid + n_val :]

x_train, y_train = x_all[train_idx], y_all[train_idx]
x_val, y_val = x_all[val_idx], y_all[val_idx]
x_test, y_test = x_all[test_idx], y_all[test_idx]

x_dense = np.linspace(-4.0, 4.0, 400)
y_clean_dense = 2.0 * np.sin(2.2 * x_dense) * np.exp(-0.15 * x_dense**2) + 0.4 * np.cos(5.5 * x_dense) + 0.3 * x_dense

# -------------------------------------------------------------
# -------------------------------------------------------------
def tricube_kernel(distances, max_dist):

    u = np.clip(distances / max_dist, 0, 1)
    return (1 - u**3)**3

def loess_single_point(x_eval, x_tr, y_tr, span=0.25, degree=1):

    distances = np.abs(x_tr - x_eval)
    k = max(int(np.ceil(span * len(x_tr))), 2)
    max_dist = np.partition(distances, k - 1)[k - 1]
    
    weights = tricube_kernel(distances, max_dist)
    
    if degree == 1:
        X = np.column_stack([np.ones_like(x_tr), x_tr - x_eval])
    else:
        X = np.column_stack([np.ones_like(x_tr), x_tr - x_eval, (x_tr - x_eval)**2])
        
    W = np.diag(weights)
    
    try:
        beta = np.linalg.lstsq(X.T @ W @ X, X.T @ W @ y_tr, rcond=None)[0]
        y_pred = beta[0]
    except np.linalg.LinAlgError:
        y_pred = np.mean(y_tr)
        beta = [y_pred, 0]
        
    return y_pred, weights, beta, max_dist

def predict_loess_full(x_eval_arr, x_tr, y_tr, span=0.25, degree=1):

    preds = [loess_single_point(x, x_tr, y_tr, span, degree)[0] for x in x_eval_arr]
    return np.array(preds)

# -------------------------------------------------------------
# -------------------------------------------------------------
spans_to_test = [0.06, 0.12, 0.22, 0.38, 0.55]
val_mses = []
for s in spans_to_test:
    val_preds = predict_loess_full(x_val, x_train, y_train, span=s)
    val_mses.append(mean_squared_error(y_val, val_preds))

best_span = spans_to_test[np.argmin(val_mses)]

# -------------------------------------------------------------
# -------------------------------------------------------------
frames_sliding = 60
total_frames = 10 + 10 + frames_sliding + 20 + 20

fig, ax = plt.subplots(figsize=(11, 6.5), dpi=100)

def animate(frame_idx):
    ax.clear()

    ax.scatter(x_train, y_train, color='blue', alpha=0.35, s=25, label='Train Data (70%)', zorder=3)
    ax.scatter(x_val, y_val, color='red', alpha=0.55, s=35, marker='o', label='Val Data (15%)', zorder=4)
    ax.scatter(x_test, y_test, color='gray', alpha=0.55, s=35, marker='s', label='Test Data (15%)', zorder=4)
    ax.plot(x_dense, y_clean_dense, 'k--', alpha=0.25, linewidth=1.5, label='True Function')

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    if frame_idx < 10:
        p = np.poly1d(np.polyfit(x_train, y_train, 1))
        ax.plot(x_dense, p(x_dense), color='red', linewidth=3.5, label='Global Linear Fit', zorder=6)
        ax.set_title("Stage 1: Global Linear Regression (High Bias)", fontsize=13, fontweight='bold', color='darkred')
        
        val_mse = mean_squared_error(y_val, p(x_val))
        info_str = f"Stage: Global Linear Fit\nVal MSE: {val_mse:.4f}\nStatus: Severe Underfitting"
        box_bg, edge_c = '#f8d7da', 'darkred'

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    elif 10 <= frame_idx < 20:
        x_eval = -0.5
        y_p, weights, beta, r_val = loess_single_point(x_eval, x_train, y_train, span=0.22)
        
        ax.axvline(x=x_eval, color='darkorange', linestyle='--', linewidth=2, label='Evaluation Point ($x_0$)', zorder=5)
        ax.axvspan(x_eval - r_val, x_eval + r_val, color='orange', alpha=0.15, label='Local Neighborhood Window')
        
        ax.scatter(x_train, y_train, c=weights, cmap='YlOrRd', s=20 + weights*110, zorder=6)
        
        ax.set_title("Stage 2: Local Weighted Regression (LOESS Concept)", fontsize=13, fontweight='bold', color='darkorange')
        info_str = (
            f"Concept: Local Weighting around $x_0 = {x_eval:.1f}$\n"
            f"Kernel: Tricube $W(u) = (1 - |u|^3)^3$\n"
            f"Status: Fitting WLS in Local Neighborhood"
        )
        box_bg, edge_c = '#fff3cd', 'darkorange'

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    elif 20 <= frame_idx < (20 + frames_sliding):
        slide_step = frame_idx - 20
        eval_indices = np.linspace(0, len(x_dense) - 1, frames_sliding, dtype=int)
        curr_idx = eval_indices[slide_step]
        x_eval = x_dense[curr_idx]
        
        y_p, weights, beta, r_val = loess_single_point(x_eval, x_train, y_train, span=0.22)
        
        x_history = x_dense[:curr_idx+1]
        y_history = predict_loess_full(x_history, x_train, y_train, span=0.22)
        ax.plot(x_history, y_history, color='darkgreen', linewidth=3.5, label='Constructed LOESS Curve', zorder=7)
        
        ax.axvline(x=x_eval, color='darkorange', linestyle='--', linewidth=1.8, zorder=5)
        ax.axvspan(x_eval - r_val, x_eval + r_val, color='orange', alpha=0.15)
        

        x_loc = np.linspace(max(-4, x_eval - r_val), min(4, x_eval + r_val), 30)
        y_loc = beta[0] + beta[1] * (x_loc - x_eval)
        ax.plot(x_loc, y_loc, color='red', linewidth=3.0, zorder=8, label='Local Linear Fit')
        
        ax.scatter(x_train, y_train, c=weights, cmap='YlOrRd', s=15 + weights*85, zorder=6)
        
        ax.set_title(f"Stage 3: Sliding Window Fitting | Point $x_0 = {x_eval:.2f}$", fontsize=13, fontweight='bold', color='darkorange')
        info_str = (
            f"Active Evaluation Point $x_0$: {x_eval:.2f}\n"
            f"Local Estimate $Y$: {y_p:.3f}\n"
            f"Status: Sliding & Fitting Local Weighted Least Squares"
        )
        box_bg, edge_c = '#fff3cd', 'darkorange'

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    elif (20 + frames_sliding) <= frame_idx < (20 + frames_sliding + 20):
        span_idx = (frame_idx - (20 + frames_sliding)) // 4
        span_idx = min(span_idx, len(spans_to_test) - 1)
        curr_span = spans_to_test[span_idx]
        
        y_dense_span = predict_loess_full(x_dense, x_train, y_train, span=curr_span)
        val_mse_curr = mean_squared_error(y_val, predict_loess_full(x_val, x_train, y_train, span=curr_span))
        
        ax.plot(x_dense, y_dense_span, color='purple', linewidth=3.5, label=f'LOESS (Span = {curr_span:.2f})', zorder=7)
        ax.set_title(f"Stage 4: Bandwidth Tuning | Span = {curr_span:.2f}", fontsize=13, fontweight='bold', color='purple')
        
        status_txt = "Overfitting (High Variance)" if curr_span < 0.15 else ("Underfitting (High Bias)" if curr_span > 0.35 else "Optimal Balance")
        info_str = (
            f"Neighborhood Span Ratio: {curr_span:.2f}\n"
            f"Validation MSE: {val_mse_curr:.4f}\n"
            f"Status: {status_txt}"
        )
        box_bg, edge_c = '#e2d9f3', 'purple'

    # ---------------------------------------------------------
    # ---------------------------------------------------------
    else:
        y_dense_best = predict_loess_full(x_dense, x_train, y_train, span=best_span)
        tr_mse = mean_squared_error(y_train, predict_loess_full(x_train, x_train, y_train, span=best_span))
        v_mse = mean_squared_error(y_val, predict_loess_full(x_val, x_train, y_train, span=best_span))
        ts_mse = mean_squared_error(y_test, predict_loess_full(x_test, x_train, y_train, span=best_span))
        
        ax.plot(x_dense, y_dense_best, color='darkgreen', linewidth=4.0, label=f'Best LOESS Model (Span={best_span:.2f})', zorder=8)
        ax.set_title("★ FINAL MODEL ★ Optimal Local Regression (LOESS/LOWESS)", fontsize=12, fontweight='bold', color='darkgreen')
        
        info_str = (
            f"FINAL LOESS MODEL (Span = {best_span:.2f})\n"
            f"Train MSE: {tr_mse:.4f} | Val MSE: {v_mse:.4f}\n"
            f"--> TEST MSE: {ts_mse:.4f} <--\n"
            f"Status: Superior Non-Parametric Fit!"
        )
        box_bg, edge_c = '#d1e7dd', 'darkgreen'

    ax.set_ylim(-3.5, 4.5)
    ax.set_xlabel('X', fontsize=11)
    ax.set_ylabel('Y', fontsize=11)
    ax.legend(loc='upper right', fontsize=8.5, framealpha=0.95, edgecolor='gray')
    ax.grid(True, linestyle='--', alpha=0.4)

    ax.text(0.48, 0.04, info_str, transform=ax.transAxes, fontsize=10.0, fontweight='bold',
            verticalalignment='bottom', bbox=dict(boxstyle='round,pad=0.5', facecolor=box_bg, alpha=0.95, edgecolor=edge_c, linewidth=2.0))

# -------------------------------------------------------------
# -------------------------------------------------------------
ani = animation.FuncAnimation(fig, animate, frames=total_frames, interval=100, repeat=True)
writer = animation.FFMpegWriter(fps=10, bitrate=2500)

# نام فایل خروجی مجزا
output_filename = 'local_regression_loess_demo.mp4'

ani.save(output_filename, writer=writer)
plt.close()

print(f"ویدیو با موفقیت ساخته و با نام '{output_filename}' ذخیره شد.")

ویدیو با موفقیت ساخته و با نام 'local_regression_loess_demo.mp4' ذخیره شد.
